## Practicum Vision & Language NLP - Uncertainty Quantification with Multimodal LLMs

In this notebook, we will evaluate several multimodal Vision & Language LLMs on a Medical Visual Question Answering task. We will use the following dataset: [GMAI-MMBench: A Comprehensive Multimodal Evaluation Benchmark Towards General Medical AI.](https://huggingface.co/datasets/OpenGVLab/GMAI-MMBench) The dataset is released under the Apache 2.0 license, and you will work on a pre-processed subset prepared by us.

We will evaluate several models:
- `Qwen2-VL-2B-Instruct` and `Medgemma-4b-it` (smaller models that can be run directly on Colab; output files are also provided in case of issues).
- `Qwen2.5-VL-32B-Instruct` and `GPT-5-mini` (larger models - we provide the output files).

Output files are named according to the following template:
`outputs_{model_name}-sample_{sample_answers}-temp_{temperature}.tsv`

### Setup

- In Runtime, select Change runtime type, then choose **T4 GPU** as the hardware accelerator.  

- Log into Hugging Face and accept the license agreement for [Qwen2-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct) and [Medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it). Attention, this may take a while to be processed.

- Set your Hugging Face token as the variable **HF_TOKEN** in the Secrets tab (left menu, key icon) or paste your token in the "your_token" in the required field. Make sure not to accidentaly share it.


### Important Note

This challenging assignment gives you plenty of freedom to explore and be creative. You are expected to demonstrate independence, formulate your own hypotheses, and test them empirically. Since this is a cutting-edge research area with many open questions, take the opportunity to investigate, experiment, and follow your curiosity! :)


In [1]:
from typing import List, Dict, Any, Tuple, Generator, Optional, Union
import random
import string, os, sys, glob
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm
import torch
import ast

In [2]:
import os
from dotenv import load_dotenv

# Load variables from the .env file into the environment
load_dotenv()

# Retrieve the token
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found. Please check your .env file.")

os.environ["HF_TOKEN"] = hf_token

In [4]:
!gdown --folder https://drive.google.com/drive/folders/1dNoFV9GvDhMdTjM2P2oHW00-f9oPMxnf -O ./data
!unzip -q ./data/filtered_images.zip -d ./data/images/

Using cookies from /Users/githawijbenga/.cache/gdown/cookies.txt
Retrieving folder contents
Retrieving folder 1MxSEpKZWI26Gl2dVdvRd2uMn7Eimr9dC model_responses
Processing file 1tG8zYy11gXQ-puRQocji6OOKogHR12Pi outputs_gpt5-mini_no-logprobs.tsv
Processing file 1GVFn2qKdDJxtEbbrufC0W7eULvAZMyPa outputs_medgemma-4b-it-temp_0.2.tsv
Processing file 1BAx2jUeuUbgRX6joWR94vdS0nsqCNf-I outputs_medgemma-4b-it-temp_0.6.tsv
Processing file 1JK7ACAkPVKIyEZ39RtVru3GDgUvMk_8J outputs_medgemma-4b-it-temp_1.0.tsv
Processing file 18cSUZl93yG167twxUY_gmEaMYXDOfi3q outputs_Qwen2-VL-2B-Instruct-temp_0.2.tsv
Processing file 1pxUlsk9rK67Uo9mOSxKKOebMzGxPYXS9 outputs_Qwen2-VL-2B-Instruct-temp_0.6.tsv
Processing file 1WlgSroLtv5STSufSC4qRM419KW5YRBod outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv
Processing file 1HjbP_JjNq1gkqsz-RiXff-c5M_tvPvb8 outputs_Qwen2.5-VL-32B-Instruct-temp_0.2.tsv
Processing file 11XjxhRI7j-xOEtnTYnE7-v8pFTOk6jIh outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv
Processing file 1FSfXTQj2ytMsCfO

# 1. Inference (run on GPUs)

## Load and Prepare Model Inputs

In [5]:
def add_options_dict(df_raw: pd.DataFrame) -> pd.DataFrame:  # prepare the multiple-choice question format
    option_cols = [col for col in df_raw.columns if col in list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")]

    def _make_dict(row):
        return {col: row[col] for col in option_cols if pd.notna(row[col])}

    out = df_raw.copy()
    out["options"] = out.apply(_make_dict, axis=1)
    return out

def construct_prompt(row) -> str:  # prepare the prompt for the model
    PROMPT_TEMPLATE = """Question: ({question})
Provide a concise answer and start your response with the letter of the selected option in square brackets. Options:
(
{options}
)
Your Answer:
"""
    question = row["question"]
    options = {
        letter: row[letter]
        for letter in string.ascii_uppercase[:5]
        if letter in row and row[letter] and str(row[letter]).strip() and not pd.isna(row[letter])
    }

    formatted_options = "\n".join(
        f"[{letter}] {option}"
        for letter, option in options.items()
    )

    return PROMPT_TEMPLATE.format(
        question=question,
        options=formatted_options
    )

In [6]:
input_tsv = "data/GMAI_mm_bench_dataset.tsv"  # the pre-processed dataset we provide
df = pd.read_csv(input_tsv, sep="\t")

df = add_options_dict(df)
df["prompt"] = df.apply(lambda row: construct_prompt(row), axis=1)
data_records: List[dict] = df.to_dict(orient="records")

In [7]:
df.head()  # inspect the data

,index,question,A,B,C,D,E,answer,category,clinical VQA task,department,perceptual granularity,modality,original task,image_path,options,prompt
0,398,Which of the following options best matches th...,fascia uncovered kidney parenchyma,lung,fascia covered kidney parenchyma,heart,clavicles,D,heart,Organ Recognition - Thorax,Cardiovascular Surgery,Contour level,X-ray,SEG_2D,398.jpg,"{'A': 'fascia uncovered kidney parenchyma', 'B...",Question: (Which of the following options best...
1,4322,"Observe the image, please select the option th...",grade 2 (minimal),grade 3 (moderate),grade 4 (severe),grade 1 (doubtful),NaN,A,grade 2 (minimal),Severity Grading,Sports Medicine,Image level,X-ray,CLS_2D,4322.jpg,"{'A': 'grade 2 (minimal)', 'B': 'grade 3 (mode...","Question: (Observe the image, please select th..."
2,2627,Considering the symptoms visible in the pictur...,COVID 19,viral pneumonia,normal chest x ray,NaN,NaN,B,viral pneumonia,Disease Diagnosis,Pulmonary Medicine,Image level,X-ray,CLS_2D,2627.jpg,"{'A': 'COVID 19', 'B': 'viral pneumonia', 'C':...",Question: (Considering the symptoms visible in...
3,70,Focus on the square-highlighted area of this X...,pulmonary mass,diffuse pulmonary nodule,pleural effusion,pulmonary calcification,NaN,A,pulmonary mass,Disease Diagnosis,Pulmonary Medicine,Box level,X-ray,DET_2D,70.jpg,"{'A': 'pulmonary mass', 'B': 'diffuse pulmonar...",Question: (Focus on the square-highlighted are...
4,4525,Choose the description that best matches the i...,negative for pneumonia,typical appearance of COVID-19,indeterminate appearance of COVID-19,atypical appearance of COVID-19,NaN,C,indeterminate appearance of COVID-19,Severity Grading,Pulmonary Medicine,Image level,X-ray,CLS_2D,4525.jpg,"{'A': 'negative for pneumonia', 'B': 'typical ...",Question: (Choose the description that best ma...


In [8]:
len(df) # we have 300 image-question pairs

300

In [ ]:
# data exploration
print(df['clinical VQA task'].value_counts())

print(df['department'].value_counts())

clinical VQA task
Disease Diagnosis                    182
Organ Recognition - Thorax            25
Bone                                  22
Severity Grading                      16
Organ Recognition - Abdomen           14
Organ Recognition - Head and Neck     13
Blood Vessels Recognition             10
Muscle                                 7
Organ Recognition - Pelvic             7
Attribute Recognition                  4
Name: count, dtype: int64
department
Pulmonary Medicine                            76
Sports Medicine                               50
Orthopedic Surgery                            49
Gastroenterology and Hepatology               32
Oncology (Medical)                            17
Cardiovascular Surgery                        14
Hematology                                    14
Otolaryngology (ENT)/Head and Neck Surgery     9
Urology                                        8
Neurosurgery                                   7
Endocrinology                                

In [34]:
# average length
df['word_count'] = df['question'].apply(lambda x: len(str(x).split()))
average_length = df['word_count'].mean()
print("average length of questions:", average_length)

# direct phrased with question mark
print("direct phrased with question mark:", df['question'].str.contains(r'\?$').sum())

# frequency of number of answer options
df['num_options_count'] = df[['A', 'B', 'C', 'D', 'E']].notna().sum(axis=1)
num_options_count = df['num_options_count'].value_counts().sort_index()

print("frequency of number of answer options:\n", num_options_count)

# gold answer letter distribution
print("gold answer letter distribution:\n", df['answer'].value_counts().sort_index())

# baseline 1: expected accuracy of a model that picks an option uniformly at random
print("baseline 1: expected accuracy of a model that picks an option uniformly at random:", (1 / df['num_options_count'].mean()))

# baseline 2: expected accuracy of a model that always picks the most frequent gold letter
most_frequent_gold_letter = df['answer'].value_counts().idxmax()
print("baseline 2: expected accuracy of a model that always picks the most frequent gold letter:", (df['answer'] == most_frequent_gold_letter).mean())    

average length of questions: 14.316666666666666
direct phrased with question mark: 274
frequency of number of answer options:
 num_options_count
2      7
3      1
4    210
5     82
Name: count, dtype: int64
gold answer letter distribution:
 answer
A    67
B    75
C    70
D    69
E    19
Name: count, dtype: int64
baseline 1: expected accuracy of a model that picks an option uniformly at random: 0.2367797947908445
baseline 2: expected accuracy of a model that always picks the most frequent gold letter: 0.25


In [58]:
# example question + answer
print(df['question'].iloc[68])
print(df.iloc[68][['A', 'B', 'C', 'D', 'E']])
print(df['answer'].iloc[68])

Please identify the part shown in the image.
A          proximal interphalangeal
B    first proximal interphalangeal
C                            radius
D               metacarpophalangeal
E                               NaN
Name: 68, dtype: object
B


## a. Inference with Qwen2-VL-2B-Instruct

In [9]:
from transformers import AutoModelForImageTextToText, AutoProcessor, GenerationConfig
from PIL import Image

model_id = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
).eval()

/Users/githawijbenga/opt/anaconda3/envs/medicalai_mps/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 729/729 [00:00<00:00, 18898.17it/s]


In [ ]:
model.device # verify this is running on 'cuda' (MPS on macbook)

device(type='mps', index=0)

In [11]:
images_dir = "data/images"

def format_inputs(batch_item: Dict[str, Any]) -> Dict[str, Any]:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": batch_item["prompt"]},
                {"type": "image"},
            ],
        }
    ]

    return {
        **batch_item,
        "messages": messages,
        "image": Image.open(os.path.join(images_dir, batch_item["image_path"]))
    }

In [59]:
batch_size = 4
max_new_tokens = 32
temperature = 0.6  # <-- Pay close attention to this parameter: you will be asked to explore how changing it affects the results.
top_p = 0.9
sample_answers = 10  # we sample 10 responses for each question.

# Repeat each element in data list 10 times
data = [item for item in data_records for _ in range(sample_answers)]

#if the block takes too long to run, use a subset for initial inspection
#data = data[:100]

results = []
for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    batch = [format_inputs(item) for item in batch]

    prompts_batch = [processor.apply_chat_template(item["messages"], add_generation_prompt=True) for item in batch]
    images_batch = [item["image"] for item in batch]

    inputs = processor(
        images=images_batch,
        text=prompts_batch,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False if temperature == 0 else True,
            temperature=temperature,
            top_p=top_p,
            return_dict_in_generate=True,
            output_scores=True,
            output_logits=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    input_ids = inputs["input_ids"]

    for b_idx, item in enumerate(batch):
        input_len = input_ids[b_idx].shape[0]

        generated_tokens = outputs.sequences[b_idx, input_len:]
        generated_text = processor.decode(generated_tokens, skip_special_tokens=True)

        logprobs = []
        for j, logits in enumerate(outputs.logits):  # extract the probability associated to each generated token
            if j >= generated_tokens.shape[0]:  # Avoid out-of-bounds if gen shorter than scores
                break
            token_id = outputs.sequences[b_idx, input_len + j].item()
            probs = torch.nn.functional.softmax(logits[b_idx], dim=-1)
            logprob = torch.log(probs[token_id]).item()
            token_str = processor.tokenizer.decode([token_id])
            logprobs.append((token_str, logprob))

        row = df[df["index"] == item["index"]].iloc[0]

        results.append({
            "index": item["index"],
            "question": item["question"],
            "gold_label": item["answer"],
            "category": item["category"],
            'clinical VQA task': item['clinical VQA task'],
            'department': item['department'],
            'perceptual granularity': item['perceptual granularity'],
            'modality': item['modality'],
            'original task': item['original task'],
            'image_path':item['image_path'],
            "options": row["options"],
            "prompt": prompts_batch[b_idx],
            "response": generated_text,
            "logprobs": logprobs
        })

# save the generated responses
output_df = pd.DataFrame(results)
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

filename = f"outputs_{os.path.split(model_id)[1]}-temp_{temperature}.tsv"
output_df.to_csv(os.path.join(output_folder, filename), index=False, sep="\t", encoding="utf-8")

100%|██████████| 750/750 [17:32<00:00,  1.40s/it]


In [97]:
import math

# investingating the output
print("response:", output_df.iloc[0]['response'])

print("token-log probability pairs:", output_df.iloc[0]['logprobs'])

# token log probability of first token into probability
print("token log probability of first token into probability:", math.exp(output_df.iloc[0]['logprobs'][0][1]))



response: [B] lung
token-log probability pairs: [('[B', -0.6402887105941772), (']', -0.0009353361674584448), (' lung', -0.006684181746095419), ('<|im_end|>', -0.00011325523519190028)]
token log probability of first token into probability: 0.5271402111077987


## b. Inference with medgemma-4b-it

In [ ]:
import gc

# Release the previous model and tensors from its inference loop.
for name in (
    "model", "processor", "inputs", "outputs", "input_ids",
    "generated_tokens", "logits", "probs"
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, GenerationConfig
from PIL import Image
import requests

model_id = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
model.device

In [ ]:
images_dir = "data/images"

def format_inputs(batch_item: Dict[str, Any]) -> Dict[str, Any]:
    image = Image.open(os.path.join(images_dir, batch_item["image_path"]))
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": batch_item["prompt"]},
                {"type": "image", "image":image},
            ],
        }
    ]

    return {
        **batch_item,
        "messages": messages,
        "image": [image],
    }

In [ ]:
batch_size = 1
max_new_tokens = 32
temperature = 0.6  # <-- Pay close attention to this parameter: you will be asked to explore how changing it affects the results.
top_p = 0.9
sample_answers = 10

# Repeat each element in data list 10 times
data = [item for item in data_records for _ in range(sample_answers)]

#if the block takes too long to run, use a subset for initial inspection
#data = data[:100]

results = []
for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    batch = [format_inputs(item) for item in batch]

    prompts_batch = [processor.apply_chat_template(item["messages"], add_generation_prompt=True) for item in batch]
    images_batch = [item["image"] for item in batch]

    inputs = processor(
        text=prompts_batch,
        images=images_batch,
        padding=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False if temperature == 0 else True,
            temperature=temperature,
            top_p=top_p,
            return_dict_in_generate=True,
            output_scores=True,
            output_logits=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    input_ids = inputs["input_ids"]

    for b_idx, item in enumerate(batch):
        input_len = input_ids[b_idx].shape[0]

        generated_tokens = outputs.sequences[b_idx, input_len:]
        generated_text = processor.decode(generated_tokens, skip_special_tokens=True)

        logprobs = []
        for j, logits in enumerate(outputs.scores):
            if j >= generated_tokens.shape[0]:
                break
            token_id = outputs.sequences[b_idx, input_len + j].item()
            probs = torch.nn.functional.softmax(logits[b_idx], dim=-1)
            logprob = torch.log(probs[token_id]).item()
            token_str = processor.tokenizer.decode([token_id])
            logprobs.append((token_str, logprob))

        row = df[df["index"] == item["index"]].iloc[0]

        results.append({
            "index": item["index"],
            "question": item["question"],
            "gold_label": item["answer"],
            "category": item["category"],
            'clinical VQA task': item['clinical VQA task'],
            'department': item['department'],
            'perceptual granularity': item['perceptual granularity'],
            'modality': item['modality'],
            'original task': item['original task'],
            'image_path': item['image_path'],
            "options": row["options"],
            "prompt": prompts_batch[b_idx],
            "response": generated_text,
            "logprobs": logprobs
        })

# save the generated responses
output_df = pd.DataFrame(results)
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

filename = f"outputs_{os.path.split(model_id)[1]}-temp_{temperature}.tsv"
output_df.to_csv(os.path.join(output_folder, filename), index=False, sep="\t", encoding="utf-8")

## Compute and Save Embeddings of Model Outputs
In Section 2, you will compute and evaluate several Uncertainty Estimation (UE) methods. One of these is Radial Dispersion Score (RDS), an embedding-based UE method. To prepare for this, we first need to compute and save the embeddings of all model responses.
We will:

1. Compute and save the embeddings for all model responses.
2. Zip the folder containing the embeddings.
3. Download the zip file.
4. Change the runtime to CPU (see Section 2) and upload the zip file to the runtime.

In [ ]:
# Load the embedding model
from sentence_transformers import SentenceTransformer
import numpy as np
import os

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
os.makedirs("/data/embeddings", exist_ok=True)
embeddings_folder = "/data/embeddings"
outputs_folder = "/data/model_responses/"

def embed_save_outputs(df, output_path, embedding_model):
    sentences = df["response"].tolist()
    sentences = [str(s) for s in sentences]
    embeddings = embedding_model.encode(sentences, batch_size=32, show_progress_bar=True)
    np.save(output_path, embeddings)

for path in glob.glob(os.path.join(outputs_folder, "*")):
    filename = os.path.basename(path)
    filename = filename.replace("outputs_", "").replace(".tsv", "")
    embedding_file = os.path.join(embeddings_folder, f"embeddings_{filename}.npy")
    df = pd.read_csv(path, sep='\t')
    if not os.path.exists(embedding_file):
        embed_save_outputs(df, embedding_file, embedding_model)

In [ ]:
# We will zip the /data/embeddings folder, download the zip file manually, since restarting the runtime will delete the results.
! zip -r /data/embeddings.zip /data/embeddings

# 2. Load and Analyze Outputs (run on CPUs) - Uncertainty Quantification


In [ ]:
import re
from collections import Counter
import ast
import numpy as np
import pandas as pd

### a. Utilities and Implementations for
* Answer extraction, accuracy, majority voting
* Uncertainty Estimation (UE) methods:
  * **Token-level probability** (log-likelihood): the probability assigned to each generated token in a response, used as a proxy for model uncertainty. Higher probability = higher confidence. Note: this is not available for `GPT-5-mini`. You have to implement this method (see the assignment report).
  * **Semantic Entropy**: measures the consistency of multiple responses to the same question. More consistent responses across runs = higher model confidence.
  * **Radial Dispersion Score (RDS)**: measures the dispersion of embeddings from their centroid, for multiple responses to the same question. You have to implement this method (see the assignment report). Note: Don't forget to upload the zip file with the saved embeddings, and unzip it!
* Uncertainty Estimation evaluation metrics:
  * Discrimination metric: Area Under the Receiver Operating Characteristic Curve (**AUROC**)
  * Calibration metrics: Expected Calibration Error (**ECE**) and **Brier Score**.


In [ ]:
def parse_response(response):
    """Extract the letter choice from the response."""
    # Look for pattern like [A], [B], etc.
    match = re.search(r'\[([A-E])\]', response)
    if match:
        return match.group(1)
    return None

def parse_options(options_str):
    """Parse the options string to get the number of options."""
    try:
        # Convert string representation of dict to actual dict
        options_dict = ast.literal_eval(options_str)
        return len(options_dict)
    except:
        # Fallback: assume 5 options if parsing fails
        return 5

def load_and_process_data(file_path):
    """Load the TSV file and process the data."""
    df = pd.read_csv(file_path, sep='\t')

    # Parse responses to extract letter choices
    df['parsed_response'] = df['response'].apply(parse_response)

    # Parse options to get number of choices per question
    df['num_options'] = df['options'].apply(parse_options)

    return df

def compute_accuracy_stats(df):
    """Compute overall accuracy and per-question statistics."""
    stats = {}

    # Overall accuracy
    correct_predictions = (df['parsed_response'] == df['gold_label']).sum()
    total_predictions = len(df)
    overall_accuracy = correct_predictions / total_predictions

    stats['overall_accuracy'] = overall_accuracy
    stats['total_samples'] = total_predictions
    stats['correct_predictions'] = correct_predictions

    # Per-question statistics (grouped by index, which is unique per question)
    question_stats = []
    unique_indices = df['index'].unique()

    for idx in unique_indices:
        question_data = df[df['index'] == idx]

        # Get all responses for this question (should be 10 samples)
        responses = question_data['parsed_response'].tolist()
        gold_label = question_data['gold_label'].iloc[0]  # Should be same for all samples
        question_text = question_data['question'].iloc[0]  # Should be same for all samples
        num_options = question_data['num_options'].iloc[0]  # Should be same for all samples

        # Calculate accuracy for this question
        correct = sum(1 for r in responses if r == gold_label)
        accuracy = correct / len(responses)

        # Calculate consistency (how often the model gives the same answer)
        response_counts = Counter(responses)
        most_common_response, most_common_count = response_counts.most_common(1)[0]
        consistency = most_common_count / len(responses)

        # Calculate entropy of responses (lower = more consistent)
        response_probs = np.array(list(response_counts.values())) / len(responses)
        entropy = -np.sum(response_probs * np.log2(response_probs))

        question_stats.append({
            'index': idx,
            'question': question_text[:50] + '...' if len(question_text) > 50 else question_text,
            'gold_label': gold_label,
            'accuracy': accuracy,
            'consistency': consistency,
            'most_common_response': most_common_response,
            'entropy': entropy,
            'num_samples': len(responses),
            'num_options': num_options
        })

    stats['question_stats'] = question_stats
    return stats


In [ ]:
def compute_semantic_entropy_uncertainty(df):
    """
    Compute semantic entropy as uncertainty metric and evaluate it.
    For each question, semantic entropy measures how uncertain the model is.
    """
    grouped = df.groupby('index')

    uncertainties = []
    correctness = []  # Binary: 1 if majority answer is correct, 0 otherwise
    num_options_list = []  # Track number of options for each question

    for idx, group in grouped:
        responses = group['parsed_response'].tolist()
        gold_label = group['gold_label'].iloc[0]
        num_options = group['num_options'].iloc[0]

        # Count responses
        response_counts = Counter(responses)
        total_responses = len(responses)

        # Calculate semantic entropy (uncertainty)
        response_probs = np.array(list(response_counts.values())) / total_responses
        semantic_entropy = -np.sum(response_probs * np.log2(response_probs))

        # Determine if majority answer is correct
        most_common_response = response_counts.most_common(1)[0][0]
        is_correct = 1 if most_common_response == gold_label else 0

        uncertainties.append(semantic_entropy)
        correctness.append(is_correct)
        num_options_list.append(num_options)

    return np.array(uncertainties), np.array(correctness), np.array(num_options_list)


def compute_calibration_metrics(uncertainties, correctness, n_bins=10):
    """Compute AUROC, ECE, and Brier score with dynamic normalization."""

    from sklearn.metrics import roc_auc_score

    # AUROC: How well does uncertainty predict incorrectness?
    # Higher uncertainty should predict lower correctness
    try:
        auroc = roc_auc_score(1 - correctness,
                              uncertainties)  # 1-correctness because higher uncertainty should predict incorrectness
    except ValueError:
        auroc = 0.5  # If all samples have same correctness

    # Expected Calibration Error (ECE)
    def compute_ece(uncertainties, correctness, n_bins=10):
        bin_edges = np.linspace(0, np.max(uncertainties) + 1e-8, n_bins + 1)
        ece = 0

        for i in range(n_bins):
            mask = (uncertainties >= bin_edges[i]) & (uncertainties < bin_edges[i + 1])
            if np.sum(mask) > 0:
                bin_accuracy = np.mean(correctness[mask])
                bin_confidence = 1 - np.mean(uncertainties[mask])
                bin_confidence = np.clip(bin_confidence, 0, 1)  # Ensure it's between 0 and 1
                bin_size = np.sum(mask) / len(uncertainties)
                ece += bin_size * abs(bin_accuracy - bin_confidence)

        return ece

    u_min, u_max = uncertainties.min(), uncertainties.max()
    if u_max > u_min:
        uncertainties = (uncertainties - u_min) / (u_max - u_min)
    else:
        uncertainties = np.zeros_like(uncertainties)
    confidences = 1 - uncertainties
    confidences = np.clip(confidences, 0, 1)

    ece = compute_ece(uncertainties, correctness, n_bins)
    brier_score = np.mean((confidences - correctness) ** 2)

    return auroc, ece, brier_score

### b. Run Evaluation

In [ ]:
# Replace this with the output file corresponding to the model and decoding configuration you want to evaluate.
file_path = "./data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv"

print("Loading and processing data...")
df = load_and_process_data(file_path)

print(f"Loaded {len(df)} samples")
print(f"Number of unique questions (indices): {df['index'].nunique()}")
print(f"Samples per question: {len(df) // df['index'].nunique()}")

# Print option distribution
option_counts = df.groupby('index')['num_options'].first().value_counts().sort_index()
print(f"Distribution of number of options:")
for num_opts, count in option_counts.items():
    print(f"  {num_opts} options: {count} questions")

In [ ]:
# Basic accuracy statistics
print("\n" + "=" * 50)
print("ACCURACY ANALYSIS")
print("=" * 50)

stats = compute_accuracy_stats(df)
print(f"Overall Accuracy: {stats['overall_accuracy']:.3f}")
print(f"Correct Predictions: {stats['correct_predictions']}/{stats['total_samples']}")

# Per-question statistics
print(f"\nPer-Question Statistics:")
print("-" * 100)
print(f"{'Index':<6} {'Question':<50} {'Gold':<5} {'Opts':<7} {'Acc':<6} {'Cons':<2} {'Pred':<4} {'Entropy':<7}")
print("-" * 100)
for q_stat in stats['question_stats']:
    print(f"{q_stat['index']:<6} {q_stat['question']:<40} {q_stat['gold_label']:<4} "
          f"{q_stat['num_options']:<4} {q_stat['accuracy']:<6.3f} {q_stat['consistency']:<6.3f} "
          f"{q_stat['most_common_response']:<4} {q_stat['entropy']:<7.3f}")

In [ ]:
# Uncertainty quantification analysis
print("\n" + "=" * 60)
print("UNCERTAINTY QUANTIFICATION ANALYSIS - Semantic Entropy")
print("=" * 60)

uncertainties, correctness, num_options_array = compute_semantic_entropy_uncertainty(df)
auroc, ece, brier_score = compute_calibration_metrics(uncertainties, correctness)

print(f"\n Discrimination and Calibration Metrics:\n")
print(f" * AUROC (uncertainty vs incorrectness): {auroc:.3f}")
print(f" * Expected Calibration Error (ECE): {ece:.3f}")
print(f" * Brier Score: {brier_score:.3f}")